# Lesson 03 — Regularisation

Lessons 01 and 02 built models and fitted them as well as possible to the training data.
That turns out to be the wrong goal. What we actually want is a model that performs well
on data it has never seen, and those two aims come apart as soon as the model is flexible
enough to memorise.

This lesson is about that gap and how to control it.

1. **Overfitting**, seen directly by watching training error and test error separate.
2. The **bias and variance decomposition**, measured rather than described.
3. **Ridge**, or L2 regularisation, which shrinks the weights.
4. **Lasso**, or L1 regularisation, which sets some of them to exactly zero.
5. **Choosing $\lambda$** with a validation set and with k-fold cross validation.

Along the way this closes two loose ends. Exercise 4 of lesson 02 found that logistic
regression on separable data inflates its weights forever, and section 9 of that lesson
warned that adding higher powers leads somewhere bad. Both are resolved here.

The reference implementation is `regularization.py`, tested by `test_regularization.py`.

## Notation

| symbol | meaning |
|---|---|
| $\lambda$ | the regularisation strength, a non negative number you choose |
| $\|w\|_2^2 = \sum_j w_j^2$ | squared L2 norm, used by ridge |
| $\|w\|_1 = \sum_j \lvert w_j \rvert$ | L1 norm, used by lasso |
| training error | the cost measured on the data used to fit |
| test error | the cost measured on data held back from fitting |

$\lambda$ is a **hyperparameter**. Unlike $w$ and $b$ it is not learned by gradient
descent, because the training data always prefers $\lambda = 0$. Section 10 covers how to
pick it honestly.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from regularization import (
    polynomial_features, zscore_normalize, train_test_split, k_fold_indices,
    compute_cost_linear, compute_gradient_linear, ridge_normal_equation,
    compute_cost_logistic, compute_gradient_logistic, gradient_descent,
    soft_threshold, compute_cost_lasso, lasso_gradient_descent,
    sigmoid, RidgeRegression, LassoRegression, RegularizedLogisticRegression,
)

rng = np.random.default_rng(1)
plt.rcParams["figure.figsize"] = (6, 4)
plt.rcParams["axes.grid"] = True

## 1. A dataset with a hidden truth

Forty noisy samples of a smooth curve. The truth is $y = \sin(2.5x)$ with Gaussian noise
of standard deviation $0.25$. Half the points are held back as a **test set** that the
model never sees during fitting.

Holding data back is the only honest way to measure generalisation. Any error computed on
the data used for fitting is optimistic, and the more flexible the model the more
optimistic it gets.

In [ ]:
def truth(x):
    return np.sin(2.5 * x)


NOISE = 0.25
m = 40
x_all = np.sort(rng.uniform(-1, 1, m))
y_all = truth(x_all) + rng.normal(0, NOISE, m)

# Split on the row indices, so every polynomial degree below reuses the same partition.
order = np.random.default_rng(2).permutation(m)
idx_test, idx_train = order[:20], order[20:]

x_train, y_train = x_all[idx_train], y_all[idx_train]
x_test, y_test = x_all[idx_test], y_all[idx_test]

grid = np.linspace(-1, 1, 300)
plt.scatter(x_train, y_train, s=30, label="training set (20 points)")
plt.scatter(x_test, y_test, s=30, marker="^", label="test set (20 points)")
plt.plot(grid, truth(grid), "k--", lw=1.5, label="the truth we are hiding")
plt.xlabel("$x$"); plt.ylabel("$y$"); plt.legend(fontsize=8)
plt.title("Forty noisy samples, split in half"); plt.show()

## 2. Overfitting, watched directly

Fit polynomials of increasing degree. Each is ordinary linear regression on the feature
matrix $[x, x^2, \ldots, x^d]$, exactly as in exercise 4 of lesson 01, solved in closed
form so that convergence never confuses the picture.

In [ ]:
def fit_polynomial(degree, lam=0.0, x_fit=None, y_fit=None):
    # fit a degree d polynomial, returning a function that predicts at new x
    x_fit = x_train if x_fit is None else x_fit
    y_fit = y_train if y_fit is None else y_fit
    X = polynomial_features(x_fit, degree)
    X_scaled, mu, sigma = zscore_normalize(X)
    w, b = ridge_normal_equation(X_scaled, y_fit, lam)

    def predict(x_new):
        X_new, _, _ = zscore_normalize(polynomial_features(x_new, degree), mu, sigma)
        return X_new @ w + b

    return predict, w, b


def mse(predict, x, y):
    return float(np.mean((predict(x) - y) ** 2))


degrees = [1, 2, 3, 5, 8, 12, 15]
rows = []
for d in degrees:
    predict, w, _ = fit_polynomial(d)
    rows.append((d, mse(predict, x_train, y_train), mse(predict, x_test, y_test),
                 np.linalg.norm(w)))

print(f"{'degree':>8}{'train MSE':>12}{'test MSE':>12}{'||w||':>14}")
for d, tr, te, norm in rows:
    flag = "  <-- best test error" if te == min(r[2] for r in rows) else ""
    print(f"{d:>8}{tr:>12.4f}{te:>12.4f}{norm:>14.2f}{flag}")

Read the three columns together, because each tells a different part of the story.

**Training error falls monotonically.** It must. A degree 15 polynomial can express every
degree 3 polynomial, so it can always match and then beat the simpler model on the data it
was fitted to. Training error can therefore never tell you a model is too complex.

**Test error falls, reaches a minimum, then rises sharply.** This is the characteristic U
shape. Left of the minimum the model is too rigid to capture the real pattern, which is
**underfitting** or **high bias**. Right of it the model is capturing the noise as if it
were signal, which is **overfitting** or **high variance**.

**The weight norm explodes.** This is the mechanical signature of overfitting, and it is
the observation the whole lesson is built on. To thread a wiggly curve exactly through
scattered points, a polynomial needs enormous coefficients that nearly cancel. If large
weights are what overfitting requires, then penalising large weights should prevent it.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for d, color in [(1, "tab:green"), (3, "tab:blue"), (15, "tab:red")]:
    predict, _, _ = fit_polynomial(d)
    axes[0].plot(grid, predict(grid), color=color, lw=1.8, label=f"degree {d}")
axes[0].scatter(x_train, y_train, s=25, c="k", alpha=0.6, label="training data")
axes[0].plot(grid, truth(grid), "k--", lw=1.2, alpha=0.7, label="truth")
axes[0].set_ylim(-2, 2); axes[0].set_xlabel("$x$"); axes[0].set_ylabel("$y$")
axes[0].legend(fontsize=8); axes[0].set_title("Too rigid, about right, too flexible")

axes[1].plot([r[0] for r in rows], [r[1] for r in rows], "o-", label="training error")
axes[1].plot([r[0] for r in rows], [r[2] for r in rows], "s-", label="test error")
axes[1].set_yscale("log"); axes[1].set_xlabel("polynomial degree"); axes[1].set_ylabel("MSE")
axes[1].legend(fontsize=8); axes[1].set_title("The U shaped test error curve")

axes[2].plot([r[0] for r in rows], [r[3] for r in rows], "o-", color="tab:red")
axes[2].set_yscale("log"); axes[2].set_xlabel("polynomial degree"); axes[2].set_ylabel("$\\|w\\|$")
axes[2].set_title("Overfitting needs enormous weights")
plt.tight_layout(); plt.show()

## 3. Bias and variance, measured

The usual explanation of overfitting is that a flexible model has low bias and high
variance. Those words have precise meanings, and both can be measured directly by
repeating the whole experiment on many independently drawn training sets.

Fix a test point $x$. Across all possible training sets of a given size, the model
produces a distribution of predictions $\hat{f}(x)$. Then

- **bias** is how far the *average* prediction sits from the truth,
  $\;\mathbb{E}[\hat{f}(x)] - f(x)$,
- **variance** is how much predictions scatter around their own average,
  $\;\mathbb{E}\big[(\hat{f}(x) - \mathbb{E}[\hat{f}(x)])^2\big]$.

The expected squared error at $x$ splits exactly into three parts:

$$\boxed{\;\mathbb{E}\big[(y - \hat{f}(x))^2\big] = \underbrace{\big(\mathbb{E}[\hat{f}(x)] - f(x)\big)^2}_{\text{bias}^2} + \underbrace{\operatorname{Var}\big[\hat{f}(x)\big]}_{\text{variance}} + \underbrace{\sigma^2}_{\text{irreducible noise}}\;}$$

The third term is the noise in the labels themselves. No model can beat it, which is why
a test error of zero is not a target. Below, 300 independent training sets are drawn and
the three terms are computed and compared against directly measured test error.

In [ ]:
N_SETS, M_PER_SET = 300, 25
x_eval = np.linspace(-0.95, 0.95, 60)


def bias_variance(degree, lam):
    predictions = np.zeros((N_SETS, len(x_eval)))
    measured_errors = []
    for s in range(N_SETS):
        r = np.random.default_rng(1000 + s)
        x_s = r.uniform(-1, 1, M_PER_SET)
        y_s = truth(x_s) + r.normal(0, NOISE, M_PER_SET)
        predict, _, _ = fit_polynomial(degree, lam, x_s, y_s)
        predictions[s] = predict(x_eval)
        fresh_labels = truth(x_eval) + r.normal(0, NOISE, len(x_eval))
        measured_errors.append(np.mean((predictions[s] - fresh_labels) ** 2))

    mean_prediction = predictions.mean(axis=0)
    bias_squared = float(np.mean((mean_prediction - truth(x_eval)) ** 2))
    variance = float(np.mean(predictions.var(axis=0)))
    return bias_squared, variance, float(np.mean(measured_errors)), predictions


settings = [("degree 1", 1, 0.0), ("degree 3", 3, 0.0), ("degree 12", 12, 0.0),
            ("degree 12, lam=0.1", 12, 0.1), ("degree 12, lam=10", 12, 10.0)]
results = {}

print(f"{'model':>22}{'bias^2':>12}{'variance':>12}{'noise':>10}{'sum':>12}{'measured':>12}")
for name, d, lam in settings:
    b2, var, measured, preds = bias_variance(d, lam)
    results[name] = (b2, var, preds)
    print(f"{name:>22}{b2:>12.4f}{var:>12.4f}{NOISE**2:>10.4f}{b2 + var + NOISE**2:>12.4f}{measured:>12.4f}")

The `sum` and `measured` columns agree, which is the decomposition confirmed numerically
rather than asserted. Now read the rows.

**Degree 1** has large bias and tiny variance. A straight line cannot bend, so it is
consistently wrong in the same way regardless of which training set it sees.

**Degree 3** has almost no bias and still tiny variance. It matches the shape of the truth
and is stable. This is the sweet spot.

**Degree 12 with no penalty** has enormous variance. The bias is small, meaning the model
is right *on average*, but any individual fit is wildly far from that average. Averaging
hundreds of wild curves gives something reasonable, but you only ever get one training
set, so you get one wild curve.

**Degree 12 with $\lambda = 0.1$** keeps the flexibility and collapses the variance to
roughly the same level as degree 3. This is the entire promise of regularisation: you do
not have to choose a rigid model, you can take a flexible one and control it.

**Degree 12 with $\lambda = 10$** overshoots. Variance falls further but bias climbs, so
the total gets worse again. Too much regularisation is its own failure mode.

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(17, 4))

for ax, name in zip(axes[:3], ["degree 1", "degree 12", "degree 12, lam=0.1"]):
    preds = results[name][2]
    for s in range(60):
        ax.plot(x_eval, preds[s], color="tab:blue", alpha=0.12, lw=0.8)
    ax.plot(x_eval, preds.mean(axis=0), color="tab:red", lw=2.2, label="average fit")
    ax.plot(x_eval, truth(x_eval), "k--", lw=1.6, label="truth")
    ax.set_ylim(-2.5, 2.5); ax.set_xlabel("$x$"); ax.set_title(name)
    ax.legend(fontsize=8)
axes[0].set_ylabel("prediction")

names = [s[0] for s in settings]
bias_vals = [results[n][0] for n in names]
var_vals = [results[n][1] for n in names]
positions = np.arange(len(names))
axes[3].bar(positions - 0.2, bias_vals, 0.4, label="bias$^2$")
axes[3].bar(positions + 0.2, var_vals, 0.4, label="variance")
axes[3].set_yscale("log"); axes[3].set_xticks(positions)
axes[3].set_xticklabels([n.replace(", ", "\n") for n in names], fontsize=7)
axes[3].legend(fontsize=8); axes[3].set_title("The trade off")
plt.tight_layout(); plt.show()

The first three panels show 60 of the 300 fitted curves in pale blue. Degree 1 is a tight
bundle in the wrong place, which is bias. Degree 12 is a chaotic spray, which is variance.
Degree 12 with a penalty is a tight bundle in the right place, which is what we want.

## 4. Ridge regression, the L2 penalty

Add a term to the cost that charges for large weights:

$$\boxed{\;J(w,b) = \frac{1}{2m}\left[\sum_{i=1}^{m}\big(f_{w,b}(x^{(i)}) - y^{(i)}\big)^2 + \lambda\sum_{j=1}^{n}w_j^2\right]\;}$$

The optimiser now faces a trade off it did not have before. It can reduce the first term by
fitting the data more closely, but only by growing weights that the second term charges
for. $\lambda$ sets the exchange rate. At $\lambda = 0$ nothing changes from lesson 01. As
$\lambda \to \infty$ every weight is crushed to zero and the model predicts a constant.

### Why the bias $b$ is excluded

The sum runs over $j = 1 \ldots n$, so it covers $w$ and not $b$. The weights control the
*shape* of the fit, which is what we want to keep simple. The bias controls the overall
*level*, and shrinking it towards zero would pull every prediction towards zero for no
reason. If your labels happen to be centred near 1000, penalising $b$ would be actively
harmful and would buy you nothing in return.

### The gradient, and why L2 is called weight decay

Differentiating the penalty $\frac{\lambda}{2m}\sum_j w_j^2$ with respect to $w_j$ gives
$\frac{\lambda}{m}w_j$, so

$$\frac{\partial J}{\partial w_j} = \frac{1}{m}\left[\sum_{i=1}^{m}\big(f_{w,b}(x^{(i)}) - y^{(i)}\big)x_j^{(i)} + \lambda w_j\right]
\qquad
\frac{\partial J}{\partial b} = \frac{1}{m}\sum_{i=1}^{m}\big(f_{w,b}(x^{(i)}) - y^{(i)}\big)$$

Note the bias gradient is untouched. Substituting the weight gradient into the update and
collecting terms shows what the penalty actually does:

$$w_j := \underbrace{w_j\left(1 - \frac{\alpha\lambda}{m}\right)}_{\text{shrink first}} - \underbrace{\frac{\alpha}{m}\sum_i\big(f - y\big)x_j^{(i)}}_{\text{then the usual step}}$$

Every iteration multiplies each weight by a number slightly below one before taking the
ordinary gradient step. That is why L2 regularisation is also called **weight decay**.

It also tells you when it will blow up. The factor $1 - \alpha\lambda/m$ must stay inside
$(-1, 1)$, which requires

$$\alpha < \frac{2m}{\lambda}$$

Raising $\lambda$ without lowering $\alpha$ is a reliable way to make a run diverge.

In [ ]:
X_demo = polynomial_features(x_train, 12)
X_demo_scaled, mu_demo, sigma_demo = zscore_normalize(X_demo)
m_train = len(y_train)

print(f"training rows m = {m_train}, so gradient descent needs alpha < 2m/lambda\n")
print(f"{'lambda':>10}{'stability bound':>18}{'alpha=0.1 ok?':>16}{'result':>28}")
for lam in (1.0, 10.0, 100.0, 1000.0):
    bound = 2 * m_train / lam
    with np.errstate(over="ignore", invalid="ignore"):   # divergence is the point here
        w_gd, b_gd, hist = gradient_descent(
            X_demo_scaled, y_train, np.zeros(12), 0.0, 0.1, 2000,
            compute_gradient_linear, compute_cost_linear, lam,
        )
    final = hist["cost"][-1]
    verdict = f"cost {final:.4f}" if np.isfinite(final) else "diverged to nan"
    print(f"{lam:>10}{bound:>18.3f}{str(0.1 < bound):>16}{verdict:>28}")

The prediction is exact. Once $\lambda$ passes $2m/\alpha = 400$, the run diverges.

### The closed form, and a bonus

Setting the gradient to zero gives a ridge version of the normal equation. Writing $X_b$
for $X$ with a leading column of ones, and $P$ for the identity matrix with its first
diagonal entry zeroed so the bias escapes the penalty:

$$\theta = \big(X_b^\top X_b + \lambda P\big)^{-1}X_b^\top y$$

Adding $\lambda$ to the diagonal has a second benefit that has nothing to do with
overfitting. Ordinary least squares fails outright when $X_b^\top X_b$ is not invertible,
which happens whenever there are more features than examples, or when two features are
perfectly correlated. The penalty repairs that.

In [ ]:
tiny = np.random.default_rng(5).normal(size=(5, 12))     # 5 examples, 12 features
tiny_y = np.random.default_rng(6).normal(size=5)
X_b = np.hstack([np.ones((5, 1)), tiny])
P = np.eye(13); P[0, 0] = 0.0

print("condition number of the matrix we have to invert")
print(f"  lambda = 0   : {np.linalg.cond(X_b.T @ X_b):.3e}")
print(f"  lambda = 1   : {np.linalg.cond(X_b.T @ X_b + P):.3e}")
print("\nfloat64 carries about 16 digits, so a condition number above 1e16 means no")
print("correct digits survive. numpy does not raise, it just returns a meaningless answer.")

## 5. What $\lambda$ actually does to the fit

Take the degree 12 model that overfitted badly and sweep $\lambda$.

In [ ]:
lambdas = np.logspace(-6, 3, 40)
train_curve, test_curve, norm_curve = [], [], []
coefficient_path = []

for lam in lambdas:
    predict, w, _ = fit_polynomial(12, lam)
    train_curve.append(mse(predict, x_train, y_train))
    test_curve.append(mse(predict, x_test, y_test))
    norm_curve.append(np.linalg.norm(w))
    coefficient_path.append(w)

coefficient_path = np.array(coefficient_path)
best_lam = lambdas[int(np.argmin(test_curve))]

print(f"{'lambda':>12}{'train MSE':>12}{'test MSE':>12}{'||w||':>14}")
for lam in (0.0, 1e-4, 1e-2, best_lam, 1.0, 10.0, 100.0):
    predict, w, _ = fit_polynomial(12, lam)
    tag = "  <-- best" if np.isclose(lam, best_lam) else ""
    print(f"{lam:>12.4g}{mse(predict, x_train, y_train):>12.4f}"
          f"{mse(predict, x_test, y_test):>12.4f}{np.linalg.norm(w):>14.3f}{tag}")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))

for lam, color in [(1e-6, "tab:red"), (best_lam, "tab:blue"), (100.0, "tab:green")]:
    predict, _, _ = fit_polynomial(12, lam)
    axes[0].plot(grid, predict(grid), color=color, lw=1.8, label=f"$\\lambda$ = {lam:.4g}")
axes[0].scatter(x_train, y_train, s=25, c="k", alpha=0.6, label="training data")
axes[0].plot(grid, truth(grid), "k--", lw=1.2, alpha=0.7, label="truth")
axes[0].set_ylim(-2, 2); axes[0].set_xlabel("$x$"); axes[0].set_ylabel("$y$")
axes[0].legend(fontsize=8); axes[0].set_title("Degree 12 at three penalty strengths")

axes[1].plot(lambdas, train_curve, label="training error")
axes[1].plot(lambdas, test_curve, label="test error")
axes[1].axvline(best_lam, color="gray", ls=":", label=f"best $\\lambda$ = {best_lam:.3g}")
axes[1].axhline(NOISE ** 2, color="r", ls="--", lw=0.8, label="irreducible noise")
axes[1].set_xscale("log"); axes[1].set_yscale("log")
axes[1].set_xlabel("$\\lambda$"); axes[1].set_ylabel("MSE"); axes[1].legend(fontsize=8)
axes[1].set_title("Under and over regularised, with a sweet spot")

for j in range(12):
    axes[2].plot(lambdas, coefficient_path[:, j], lw=1.2)
axes[2].axvline(best_lam, color="gray", ls=":")
axes[2].set_xscale("log"); axes[2].set_xlabel("$\\lambda$"); axes[2].set_ylabel("weight value")
axes[2].set_title("Ridge coefficient path: all 12 shrink, none reach zero")
plt.tight_layout(); plt.show()

The middle panel is the same U shape as the degree sweep, which is the point:
**increasing $\lambda$ and decreasing model complexity do the same job.** Regularisation
gives you a continuous dial where model selection only gave you integers.

The right panel is the **coefficient path**. Every weight slides smoothly towards zero as
$\lambda$ grows, and crucially none of them arrives. Ridge produces small weights, never
absent ones. That is the difference lasso addresses.

### Scaling is not optional here

The penalty $\sum_j w_j^2$ treats every weight identically, but a weight's natural size
depends on the units of its feature. A feature measured in thousands needs a weight a
thousand times smaller than one measured in units to have the same effect, so the penalty
would punish them completely differently for no principled reason. Every fit in this
notebook applies z-score normalisation first, and every fit outside it should too.

In [ ]:
x_scaled_demo = x_train * 1000.0          # same data, different units
X_a = polynomial_features(x_train, 5)
X_b_units = polynomial_features(x_scaled_demo, 5)

print("degree 5 fit, lambda = 1, with and without scaling")
for name, X_raw in [("x in original units", X_a), ("x multiplied by 1000", X_b_units)]:
    w_unscaled, b_unscaled = ridge_normal_equation(X_raw, y_train, lam=1.0)
    X_scaled, _, _ = zscore_normalize(X_raw)
    w_scaled, b_scaled = ridge_normal_equation(X_scaled, y_train, lam=1.0)
    fit_unscaled = np.mean((X_raw @ w_unscaled + b_unscaled - y_train) ** 2)
    fit_scaled = np.mean((X_scaled @ w_scaled + b_scaled - y_train) ** 2)
    print(f"  {name:<24} train MSE unscaled {fit_unscaled:8.4f}   scaled {fit_scaled:8.4f}")
print("\nWithout scaling the same lambda means something completely different in each")
print("set of units. With scaling the two runs agree, as they must.")

## 6. Lasso, the L1 penalty

Replace the squared penalty with a sum of absolute values:

$$\boxed{\;J(w,b) = \frac{1}{2m}\sum_{i=1}^{m}\big(f_{w,b}(x^{(i)}) - y^{(i)}\big)^2 + \frac{\lambda}{m}\sum_{j=1}^{n}\lvert w_j \rvert\;}$$

The change looks small and the consequence is not. Lasso drives weights to **exactly**
zero, which removes features from the model entirely. It performs feature selection as a
side effect of fitting.

### Why L1 gives exact zeros and L2 does not

Compare the two penalties near zero. The derivative of $w^2$ is $2w$, which itself goes to
zero as $w$ does, so the pressure to shrink fades away exactly when the weight gets small.
A ridge weight approaches zero and never arrives.

The derivative of $\lvert w \rvert$ is $\pm 1$, constant no matter how small $w$ is. The
pressure to shrink never weakens, so a weight whose contribution to the fit is worth less
than $\lambda$ gets pushed all the way to zero and pinned there.

There is a geometric version of the same statement. Minimising the error subject to a
budget on the weights means finding where the error contours first touch the budget
region. For L2 that region is a circle, and a curve touching a circle generically touches
it away from the axes. For L1 it is a diamond whose corners lie **on** the axes, and
corners are exactly where a contour is most likely to make first contact. A corner means
one coordinate equals zero.

In [ ]:
# a two feature least squares problem, so both penalties can be drawn
rng_geo = np.random.default_rng(3)
X_geo = rng_geo.normal(size=(30, 2))
w_true_geo = np.array([1.6, 0.45])
y_geo = X_geo @ w_true_geo + rng_geo.normal(0, 0.4, 30)

w1 = np.linspace(-0.4, 2.4, 300)
w2 = np.linspace(-1.0, 1.8, 300)
W1, W2 = np.meshgrid(w1, w2)
J_geo = np.array([[np.mean((X_geo @ np.array([a, c]) - y_geo) ** 2) / 2
                   for a in w1] for c in w2])
w_ols = np.linalg.solve(X_geo.T @ X_geo, X_geo.T @ y_geo)

budget = 0.75

# Solve each constrained problem by brute force on a fine grid, so the marked point is a
# computed answer rather than an assumed one.
fine = np.linspace(-1.2, 1.2, 1201)
G1, G2 = np.meshgrid(fine, fine)
candidates = np.column_stack([G1.ravel(), G2.ravel()])
residuals = candidates @ X_geo.T - y_geo
objective = np.mean(residuals ** 2, axis=1) / 2

def best_within(mask):
    inside = np.where(mask)[0]
    return candidates[inside[np.argmin(objective[inside])]]

solution_l2 = best_within(np.linalg.norm(candidates, axis=1) <= budget)
solution_l1 = best_within(np.abs(candidates).sum(axis=1) <= budget)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
theta = np.linspace(0, 2 * np.pi, 400)
for ax, kind, solution in [(axes[0], "L2 (ridge): a circle", solution_l2),
                           (axes[1], "L1 (lasso): a diamond", solution_l1)]:
    ax.contour(W1, W2, J_geo, levels=np.logspace(-0.9, 1.3, 14), cmap="viridis", alpha=0.75)
    ax.plot(*w_ols, "k*", markersize=14, label="unpenalised optimum")
    if kind.startswith("L2"):
        ax.plot(budget * np.cos(theta), budget * np.sin(theta), "r-", lw=2, label="budget region")
    else:
        ax.plot([budget, 0, -budget, 0, budget], [0, budget, 0, -budget, 0],
                "r-", lw=2, label="budget region")
    ax.plot(*solution, "ro", markersize=10, label="constrained solution")
    ax.axhline(0, color="gray", lw=0.6); ax.axvline(0, color="gray", lw=0.6)
    ax.set_xlabel("$w_1$"); ax.set_ylabel("$w_2$"); ax.set_title(kind)
    ax.legend(fontsize=8); ax.set_aspect("equal")
plt.tight_layout(); plt.show()

print(f"unpenalised optimum      : w = {w_ols.round(4)}")
print(f"best point inside the circle : w = {solution_l2.round(4)}   both weights nonzero")
print(f"best point inside the diamond: w = {solution_l1.round(4)}   w2 is exactly zero")

### Solving it: soft thresholding

The absolute value has no derivative at zero, so plain gradient descent cannot reach an
exact zero. It would step past and oscillate. The standard fix is **proximal gradient
descent**, also called ISTA: take an ordinary gradient step on the squared error only,
then apply the penalty exactly with a shrinkage operator.

$$\operatorname{soft}(x, t) = \operatorname{sign}(x)\max\big(\lvert x \rvert - t, 0\big)$$

Each weight is pulled towards zero by a fixed amount $t$ and clamped there if it would
overshoot. The clamping is where the exact zeros come from.

In [ ]:
values = np.linspace(-3, 3, 400)
plt.plot(values, values, "k--", lw=1, label="no shrinkage")
plt.plot(values, soft_threshold(values, 1.0), lw=2, label="soft_threshold(x, 1.0)")
plt.axhline(0, color="gray", lw=0.6); plt.axvline(0, color="gray", lw=0.6)
plt.xlabel("$x$"); plt.ylabel("output"); plt.legend(fontsize=8)
plt.title("Everything within 1.0 of zero is set to exactly zero"); plt.show()

print("soft_threshold([-5, -0.5, 0.5, 5], 1.0) =", soft_threshold(np.array([-5.0, -0.5, 0.5, 5.0]), 1.0))

In [ ]:
X_lasso, mu_l, sigma_l = zscore_normalize(polynomial_features(x_train, 12))
X_lasso_test, _, _ = zscore_normalize(polynomial_features(x_test, 12), mu_l, sigma_l)

lasso_lambdas = np.logspace(-3, 2, 30)
lasso_path, lasso_test, lasso_kept = [], [], []
for lam in lasso_lambdas:
    w_l, b_l, _ = lasso_gradient_descent(X_lasso, y_train, np.zeros(12), 0.0,
                                         0.02, 20_000, lam)
    lasso_path.append(w_l)
    lasso_test.append(float(np.mean((X_lasso_test @ w_l + b_l - y_test) ** 2)))
    lasso_kept.append(int(np.sum(np.abs(w_l) > 1e-8)))
lasso_path = np.array(lasso_path)

print(f"{'lambda':>10}{'lasso keeps':>14}{'ridge keeps':>14}{'lasso test MSE':>18}")
for lam, kept, te in zip(lasso_lambdas, lasso_kept, lasso_test):
    if lam in lasso_lambdas[::6] or lam == lasso_lambdas[-1]:
        _, w_r, _ = fit_polynomial(12, lam)
        print(f"{lam:>10.4g}{kept:>14}{int(np.sum(np.abs(w_r) > 1e-8)):>14}{te:>18.4f}")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))

for j in range(12):
    axes[0].plot(lambdas, coefficient_path[:, j], lw=1.2)
axes[0].set_xscale("log"); axes[0].set_xlabel("$\\lambda$"); axes[0].set_ylabel("weight value")
axes[0].set_title("Ridge path: smooth shrinkage, nothing reaches zero")

for j in range(12):
    axes[1].plot(lasso_lambdas, lasso_path[:, j], lw=1.2)
axes[1].set_xscale("log"); axes[1].set_xlabel("$\\lambda$"); axes[1].set_ylabel("weight value")
axes[1].set_title("Lasso path: weights hit zero and stay there")

axes[2].plot(lasso_lambdas, lasso_kept, "o-", label="lasso")
axes[2].axhline(12, color="tab:orange", ls="--", label="ridge (always 12)")
axes[2].set_xscale("log"); axes[2].set_xlabel("$\\lambda$")
axes[2].set_ylabel("weights that are not exactly zero")
axes[2].legend(fontsize=8); axes[2].set_title("Lasso selects features, ridge does not")
plt.tight_layout(); plt.show()

### Choosing between them

| | ridge (L2) | lasso (L1) |
|---|---|---|
| penalty | $\sum w_j^2$ | $\sum \lvert w_j \rvert$ |
| effect | shrinks every weight | zeroes some weights outright |
| gives a sparse model | no | yes |
| closed form solution | yes | no, needs an iterative solver |
| correlated features | shares weight between them | tends to pick one and drop the rest |
| use it when | all features plausibly matter | you suspect most features are irrelevant |

Elastic net combines both penalties and is the usual choice when you want sparsity but
have groups of correlated features that lasso would arbitrarily thin out.

## 7. Regularised logistic regression

The same penalty attaches to cross-entropy without any new ideas:

$$J(w,b) = \frac{1}{m}\sum_{i=1}^{m}\left[-y^{(i)}\log f_{w,b}(x^{(i)}) - (1 - y^{(i)})\log\big(1 - f_{w,b}(x^{(i)})\big)\right] + \frac{\lambda}{2m}\sum_{j=1}^{n}w_j^2$$

and the gradient picks up the same extra term, since the penalty does not care what the
data term was:

$$\frac{\partial J}{\partial w_j} = \frac{1}{m}\left[\sum_{i=1}^{m}\big(f_{w,b}(x^{(i)}) - y^{(i)}\big)x_j^{(i)} + \lambda w_j\right]$$

### This closes exercise 4 of lesson 02

That exercise found that on perfectly separable data the weight norm grows forever, the
cost falls towards zero without reaching it, and nothing ever converges. The cause was
that scaling $w$ and $b$ up by any factor leaves the decision boundary unmoved while
lowering the cost, so the optimiser has an incentive that never runs out.

The penalty removes that incentive. Scaling the weights up now costs $\lambda$ times a
growing quantity, so a finite optimum exists again.

In [ ]:
rng_sep = np.random.default_rng(0)
X_sep = np.vstack([rng_sep.normal([-3.0, -3.0], 0.5, size=(50, 2)),
                   rng_sep.normal([3.0, 3.0], 0.5, size=(50, 2))])
y_sep = np.concatenate([np.zeros(50), np.ones(50)])

print(f"{'lambda':>10}{'||w|| @ 20k':>14}{'||w|| @ 100k':>15}{'still growing?':>17}{'accuracy':>11}")
histories = {}
for lam in (0.0, 0.01, 0.1, 1.0, 10.0):
    w_l, b_l, _ = gradient_descent(X_sep, y_sep, np.zeros(2), 0.0, 0.5, 20_000,
                                   compute_gradient_logistic, compute_cost_logistic, lam)
    norm_20k = np.linalg.norm(w_l)
    w_l, b_l, _ = gradient_descent(X_sep, y_sep, w_l, b_l, 0.5, 80_000,
                                   compute_gradient_logistic, compute_cost_logistic, lam)
    norm_100k = np.linalg.norm(w_l)
    acc = float(np.mean((sigmoid(X_sep @ w_l + b_l) >= 0.5).astype(float) == y_sep))
    histories[lam] = (w_l, b_l)
    growing = "yes" if norm_100k > norm_20k + 1e-6 else "no, it settled"
    print(f"{lam:>10}{norm_20k:>14.4f}{norm_100k:>15.4f}{growing:>17}{acc:>11.3f}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

for lam, color in [(0.0, "tab:red"), (0.1, "tab:blue"), (10.0, "tab:green")]:
    w_track, b_track = np.zeros(2), 0.0
    marks, norms_track = [], []
    for block in range(40):
        w_track, b_track, _ = gradient_descent(X_sep, y_sep, w_track, b_track, 0.5, 2000,
                                               compute_gradient_logistic,
                                               compute_cost_logistic, lam)
        marks.append((block + 1) * 2000)
        norms_track.append(np.linalg.norm(w_track))
    axes[0].plot(marks, norms_track, color=color, label=f"$\\lambda$ = {lam}")
axes[0].set_xscale("log"); axes[0].set_xlabel("iteration"); axes[0].set_ylabel("$\\|w\\|$")
axes[0].legend(fontsize=8); axes[0].set_title("Only the unpenalised run keeps climbing")

x1_line = np.array([X_sep[:, 0].min() - 0.5, X_sep[:, 0].max() + 0.5])
axes[1].scatter(X_sep[y_sep == 0, 0], X_sep[y_sep == 0, 1], s=20, c="tab:blue", label="class 0")
axes[1].scatter(X_sep[y_sep == 1, 0], X_sep[y_sep == 1, 1], s=20, c="tab:red",
                marker="^", label="class 1")
for lam, style in [(0.0, "r-"), (1.0, "b--"), (10.0, "g-.")]:
    w_l, b_l = histories[lam]
    axes[1].plot(x1_line, -(w_l[0] * x1_line + b_l) / w_l[1], style,
                 label=f"$\\lambda$ = {lam}")
axes[1].set_xlabel("$x_1$"); axes[1].set_ylabel("$x_2$"); axes[1].legend(fontsize=8)
axes[1].set_title("All boundaries separate the data equally well")
plt.tight_layout(); plt.show()

With $\lambda = 0$ the weight norm is still climbing after 100,000 iterations. With any
$\lambda > 0$ it reaches a fixed value and stops, while training accuracy stays at
$1.000$ throughout. The boundaries in the right panel all separate the data perfectly, so
the extra weight magnitude bought nothing but overconfidence.

## 8. Choosing $\lambda$ honestly

$\lambda$ cannot be fitted on the training data, because the training error always
improves as $\lambda$ falls. It cannot be fitted on the test data either, or the test set
stops being an honest estimate of performance. The usual answer is a three way split:

- **training set**, used to fit $w$ and $b$,
- **validation set**, used to compare values of $\lambda$,
- **test set**, touched exactly once at the very end.

When data is scarce, **k-fold cross validation** reuses it. Split the training data into
$k$ parts, then for each candidate $\lambda$ fit $k$ times, each time holding out a
different part for validation, and average the $k$ scores. Every point is used for
validation exactly once and for training $k - 1$ times.

In [ ]:
def cross_validate(degree, lam, x_data, y_data, k=5, seed=0):
    # average validation MSE over k folds
    scores = []
    for train_idx, val_idx in k_fold_indices(len(y_data), k=k, seed=seed):
        predict, _, _ = fit_polynomial(degree, lam, x_data[train_idx], y_data[train_idx])
        scores.append(np.mean((predict(x_data[val_idx]) - y_data[val_idx]) ** 2))
    return float(np.mean(scores)), float(np.std(scores))


cv_lambdas = np.logspace(-5, 2, 30)
cv_means, cv_stds = [], []
for lam in cv_lambdas:
    mean_score, std_score = cross_validate(12, lam, x_train, y_train, k=5)
    cv_means.append(mean_score)
    cv_stds.append(std_score)

cv_means, cv_stds = np.array(cv_means), np.array(cv_stds)
lam_cv = cv_lambdas[int(np.argmin(cv_means))]

predict_final, w_final, _ = fit_polynomial(12, lam_cv)
print(f"cross validation picked lambda = {lam_cv:.4g}")
print(f"the test set, touched now for the first time, gives MSE "
      f"{mse(predict_final, x_test, y_test):.4f}")
print(f"the best possible test MSE over the whole sweep was {min(test_curve):.4f} "
      f"at lambda = {best_lam:.4g}")
print(f"\nthe noise variance is {NOISE ** 2:.4f}, which is the floor *in expectation*.")
print("Our measured test error dips below it, because 20 test points is a small sample")
print("and the estimate fluctuates around the true value rather than sitting on it.")

plt.errorbar(cv_lambdas, cv_means, yerr=cv_stds, fmt="o-", capsize=3, ms=4,
             label="5-fold cross validation")
plt.plot(lambdas, test_curve, "s--", ms=3, alpha=0.7, label="true test error")
plt.axvline(lam_cv, color="tab:green", ls=":", label=f"chosen $\\lambda$ = {lam_cv:.3g}")
plt.xscale("log"); plt.yscale("log")
plt.xlabel("$\\lambda$"); plt.ylabel("MSE"); plt.legend(fontsize=8)
plt.title("Cross validation locates the minimum without touching the test set")
plt.show()

Cross validation finds a $\lambda$ close to the true optimum using only the training data.

One detail worth not skipping past: the measured test error at the best $\lambda$ comes out
*below* the noise variance $\sigma^2 = 0.0625$, which the decomposition in section 3 called
a floor. There is no contradiction. The floor holds for the error averaged over infinitely
many test points. Our test set has 20, so the measured value is a noisy estimate that
scatters either side of the truth. Small held out sets give noisy scores, which is exactly
why cross validation averages over folds instead of trusting one split.
The error bars are wide, which is honest: with 20 training points each fold holds out only
4, so any single fold's estimate is noisy. That is an argument for reporting the spread
rather than only the mean, and for the common rule of choosing the largest $\lambda$ whose
score is within one standard error of the best, since simpler models generalise more
safely when the evidence cannot distinguish them.

## 9. Checking against the module

`regularization.py` provides `RidgeRegression`, `LassoRegression` and
`RegularizedLogisticRegression`, all with the same shape as the classes in lessons 01 and
02.

In [ ]:
X_train_poly = polynomial_features(x_train, 12)
X_test_poly = polynomial_features(x_test, 12)

ridge = RidgeRegression(lam=1.0, alpha=0.01, num_iters=50_000).fit(X_train_poly, y_train)
lasso = LassoRegression(lam=1.0, alpha=0.02, num_iters=50_000).fit(X_train_poly, y_train)

print(f"{'model':<10}{'train MSE':>12}{'test MSE':>12}{'R^2 (train)':>14}{'weights kept':>15}")
for name, model in [("ridge", ridge), ("lasso", lasso)]:
    print(f"{name:<10}{model.mse(X_train_poly, y_train):>12.4f}"
          f"{model.mse(X_test_poly, y_test):>12.4f}"
          f"{model.score(X_train_poly, y_train):>14.4f}"
          f"{int(np.sum(np.abs(model.w) > 1e-8)):>15} of 12")

logistic = RegularizedLogisticRegression(lam=1.0, alpha=0.5, num_iters=5000).fit(X_sep, y_sep)
print(f"\nregularised logistic on the separable data: accuracy "
      f"{logistic.score(X_sep, y_sep):.3f}, ||w|| = {np.linalg.norm(logistic.w):.4f}")

## Exercises

1. **Ridge as data augmentation.** Ridge with penalty $\lambda$ is exactly ordinary least
   squares on an enlarged dataset: append $n$ extra rows equal to $\sqrt{\lambda}\,I$ with
   targets of zero. Verify this numerically by building the augmented matrix, solving it
   with the plain normal equation from lesson 01, and comparing against
   `ridge_normal_equation`. Explain in one sentence why the trick works.

2. **Early stopping is regularisation.** Fit the degree 12 model with $\lambda = 0$ by
   gradient descent and record the test error every 100 iterations. Plot it. Show that it
   falls, reaches a minimum, and then rises, and that stopping at the minimum gives a test
   error competitive with the best $\lambda$. Explain the connection to the weight norm.

3. **The one standard error rule.** Using the cross validation output, implement the rule
   of choosing the largest $\lambda$ whose mean score is within one standard error of the
   best mean score. Compare the resulting model against the plain minimum on the test set
   and on weight norm.

4. **Lasso on genuinely irrelevant features.** Build a dataset with 5 informative features
   and 45 pure noise features. Fit ridge and lasso, and report how many of the 45 noise
   weights each drives below $10^{-8}$. Then check which model predicts better on held out
   data, and whether the features lasso kept are the informative ones.

5. **Learning curves.** For a fixed model, plot training and test error against the number
   of training examples, from 5 up to several hundred. Do it for degree 1 and for degree 15.
   Explain how the shape of the gap between the two curves tells you whether collecting
   more data will help, and why it will not help the high bias model.

## What's next

Lesson 04, **neural networks**: stack these linear-then-nonlinear building blocks into
layers, and compute the gradient through all of them with backpropagation. Everything from
these three lessons carries forward. The sigmoid becomes an activation function,
cross-entropy stays the loss for classification, weight decay stays the regulariser, and
gradient descent is still the optimiser.